In [ ]:
# import os
# import sys
# project_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))
# sys.path.append(project_path)
#
# requirements_path = os.path.join(project_path, "SECONDARY/requirements.txt")
# !{sys.executable} -m pip install -r "{requirements_path}"

In [ ]:
import os
import sys
import time

!{sys.executable} --version
if sys.version_info.minor == 8:
    raise RuntimeError('USE JUPYTER KERNEL VENV 3.10/310/DEFAULT INSTEAD')

!cd /workspace/CRYPTO_MACAQUES && pip install .
!cd /home/crypto/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && source venv/bin/activate && !{sys.executable} -m pip install .

from IPython.core.display_functions import clear_output
clear_output()

In [ ]:
%load_ext autoreload
%autoreload

from datetime import timedelta
from dateutil import parser

from SRC.LIBRARIES.new_data_utils import fetch
from SRC.CORE._CONSTANTS import _KIEV_TIMESTAMP
import SRC.LIBRARIES.new_utils as nu


# ========== TP/SL ==========
TP_LEVEL = 'near_green'           # near_green, yellow, far_green
TP_MODE = 'dynamic'               # fixed, dynamic
SL_RATIO = 1
# ========== НАСТРОЙКИ ВХОДА ==========
LOOKBACK_MIN_BARS = 0
REQUIRE_GREEN_TOUCH_AFTER_SL = False
MIN_BARS_OUTSIDE = 1
# ========== ТРЕЙЛИНГ ==========
TRAILING_TP = True
TRAILING_TP_DISTANCE = 0.5
# ========== ФИЛЬТР ПО СТОХАСТИКУ ==========
STOCH_FILTER = False               # включить/выключить фильтр
STOCH_SERIES = 'D'                 # 'K' или 'D' - какой стохастик использовать
STOCH_OVERSOLD = 20                # нижняя граница для long
STOCH_OVERBOUGHT = 80              # верхняя граница для short
# ==================================
# todo xrp 4h 2025 and 2026 champion.
# todo doge/xrp 4h 2026 champions
sym='xrp'.upper()
symbol = f'{sym}USDT'
discretization = '4h'.upper()
display_start_date_str = '2026-01-01'
display_end_date_str = '2026-05-21'
capital_per_trade = 1000
commission_rate = 0.00075

load_end_date = parser.parse(display_end_date_str)

market_type = 'SPOT'
mrc_buffer = 1000
rsi_buffer = 200
stoch_buffer = 50
macd_buffer = 200
atr_buffer = 200
display_start_dt = parser.parse(display_start_date_str)
start_time = time.perf_counter()

discretization_value = int(discretization[:-1])
timeframe_seconds = {
    'M': discretization_value * 60,
    'H': discretization_value * 3600,
    'D': discretization_value * 86400
}.get(discretization[-1], 1800)

load_start_dt = display_start_dt - timedelta(seconds=timeframe_seconds * mrc_buffer)

mrc_df = fetch(market_type, symbol, discretization, load_start_dt, load_end_date)
df = mrc_df.iloc[mrc_buffer:].copy()

In [ ]:
# Убеждаемся, что индексы уникальны
mrc_df = mrc_df[~mrc_df.index.duplicated(keep='first')]
df = df[~df.index.duplicated(keep='first')]

# 1. RSI
rsi_calc_df = nu.prepare_buffer_data(mrc_df, df, rsi_buffer)
df = nu.rsi(df, rsi_calc_df)

# 2. Stochastic
stoch_calc_df = nu.prepare_buffer_data(mrc_df, df, stoch_buffer)
df = nu.stochastic_tradingview(df, stoch_calc_df)

# 3. MACD
macd_calc_df = nu.prepare_buffer_data(mrc_df, df, macd_buffer)
df, macd = nu.macd(df, macd_calc_df)

# 4. ATR
atr_calc_df = nu.prepare_buffer_data(mrc_df, df, atr_buffer)
df = nu.atr(df, atr_calc_df)

# Расчёт MRC
mrc_df = nu.mrc_calculate(mrc_df, df)

# SMA для фильтра (период можно менять)
sma_period = 20
df['sma'] = df['close'].rolling(sma_period).mean()

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, Tuple


def backtest_red_cross(
    df: pd.DataFrame,
    commission: float = 0.00075,
    capital_per_trade: float = 1000.0,
    slippage: float = 0.0002,
    lookback_min_bars: int = 0,
    tp_level: str = 'far_green',
    sl_ratio: float = 3.0,
    tp_mode: str = 'fixed',
    trailing_tp: bool = False,
    trailing_tp_distance: float = 0.5,
    require_green_touch_after_sl: bool = False,
    min_bars_outside: int = 0,
    stoch_filter: bool = False,
    stoch_series: str = 'D',
    stoch_oversold: int = 20,
    stoch_overbought: int = 80
) -> Tuple[pd.DataFrame, Dict]:
    """
    Стратегия на обратном пересечении красной линии.

    Вход: тело свечи пересекает канал [loband2, upband2]
    при условии, что предыдущее тело было за красной линией.

    Long:  предыдущее тело ниже loband2 И текущее тело пересекает канал
    Short: предыдущее тело выше upband2 И текущее тело пересекает канал

    min_bars_outside: минимальное количество последовательных свечей,
                      в которых тело (частично) находится за красной линией
                      перед разрешением входа.
                      
    stoch_filter: включить фильтр по стохастику
    stoch_series: 'K' или 'D' - какой стохастик использовать
    stoch_oversold: нижняя граница для long (стохастик должен быть ниже)
    stoch_overbought: верхняя граница для short (стохастик должен быть выше)
    """

    df_back = df.copy()

    # ========== ПОДГОТОВКА УСЛОВИЙ ВХОДА ==========

    # Текущий close и предыдущий close
    prev_close = df_back['close'].shift(1)

    # Границы тела текущей свечи
    body_low = np.minimum(df_back['open'], df_back['close'])
    body_high = np.maximum(df_back['open'], df_back['close'])

    # Текущее тело пересекает канал (частично или полностью)
    body_touches_channel = (
        (body_high >= df_back['loband2']) & (body_low <= df_back['upband2'])
    )

    # Тело находится за красными линиями (частично)
    # Для long: тело ниже loband2
    body_below_red = (df_back['open'] < df_back['loband2']) | (df_back['close'] < df_back['loband2'])
    # Для short: тело выше upband2
    body_above_red = (df_back['open'] > df_back['upband2']) | (df_back['close'] > df_back['upband2'])

    # Предыдущее тело целиком ниже нижней красной
    prev_body_below = (
        (df_back['open'].shift(1) < df_back['loband2'].shift(1)) &
        (df_back['close'].shift(1) < df_back['loband2'].shift(1))
    )

    # Предыдущее тело целиком выше верхней красной
    prev_body_above = (
        (df_back['open'].shift(1) > df_back['upband2'].shift(1)) &
        (df_back['close'].shift(1) > df_back['upband2'].shift(1))
    )

    # Базовые условия входа (без учёта min_bars_outside)
    long_enter_base = (
        ((prev_close < df_back['loband2']) | prev_body_below) &
        body_touches_channel
    )

    short_enter_base = (
        ((prev_close > df_back['upband2']) | prev_body_above) &
        body_touches_channel
    )

    # ========== ПОДСЧЁТ СВЕЧЕЙ ЗА КРАСНОЙ ЛИНИЕЙ ==========

    # Счётчик последовательных свечей ниже loband2 (для long)
    below_counter = 0
    below_count_series = np.zeros(len(df_back))

    # Счётчик последовательных свечей выше upband2 (для short)
    above_counter = 0
    above_count_series = np.zeros(len(df_back))

    for i in range(len(df_back)):
        # Обновляем счётчик для long (тело ниже красной)
        if body_below_red.iloc[i]:
            below_counter += 1
        else:
            below_counter = 0
        below_count_series[i] = below_counter

        # Обновляем счётчик для short (тело выше красной)
        if body_above_red.iloc[i]:
            above_counter += 1
        else:
            above_counter = 0
        above_count_series[i] = above_counter

    # Добавляем счётчики в df_back для отладки (опционально)
    df_back['bars_below_red'] = below_count_series
    df_back['bars_above_red'] = above_count_series

    # Финальные условия входа с учётом min_bars_outside
    # Для входа нужно, чтобы на предыдущей свече счётчик был >= min_bars_outside
    prev_below_count = np.roll(below_count_series, 1)
    prev_below_count[0] = 0

    prev_above_count = np.roll(above_count_series, 1)
    prev_above_count[0] = 0

    long_enter = long_enter_base & (prev_below_count >= min_bars_outside)
    short_enter = short_enter_base & (prev_above_count >= min_bars_outside)

    # ========== ВЫБОР СТОХАСТИКА ==========
    if stoch_series.upper() == 'K':
        stoch_values = df_back['stoch_k']
    else:  # 'D' по умолчанию
        stoch_values = df_back['stoch_d']

    # ========== ИНИЦИАЛИЗАЦИЯ ==========
    df_back['signal'] = 0
    df_back['entry_price'] = np.nan
    df_back['exit_price'] = np.nan
    df_back['pnl'] = 0.0
    df_back['trade_type'] = ''

    position = None
    trades = []

    # ========== ОСНОВНОЙ ЦИКЛ ==========
    for i in range(1, len(df_back)):

        # Данные текущей свечи
        current_open = df_back.iloc[i]['open']
        current_close = df_back.iloc[i]['close']
        current_high = df_back.iloc[i]['high']
        current_low = df_back.iloc[i]['low']

        # MRC линии
        loband2 = df_back.iloc[i]['loband2']
        upband2 = df_back.iloc[i]['upband2']
        loband1 = df_back.iloc[i]['loband1']
        upband1 = df_back.iloc[i]['upband1']
        meanline = df_back.iloc[i]['meanline']

        # Текущее значение стохастика
        current_stoch = stoch_values.iloc[i]

        # Флаги входа
        is_long_signal = long_enter.iloc[i]
        is_short_signal = short_enter.iloc[i]

        # ========== ВХОД В ПОЗИЦИЮ ==========
        if position is None:

            direction = None
            entry_price = None
            tp_price = None
            tp_line = None

            if is_long_signal:
                direction = 'long'
                entry_price = loband2 * (1 + slippage)

                if tp_level == 'near_green':
                    tp_price = loband1
                    tp_line = 'loband1'
                elif tp_level == 'yellow':
                    tp_price = meanline
                    tp_line = 'meanline'
                else:  # far_green
                    tp_price = upband1
                    tp_line = 'upband1'

            elif is_short_signal:
                direction = 'short'
                entry_price = upband2 * (1 - slippage)

                if tp_level == 'near_green':
                    tp_price = upband1
                    tp_line = 'upband1'
                elif tp_level == 'yellow':
                    tp_price = meanline
                    tp_line = 'meanline'
                else:  # far_green
                    tp_price = loband1
                    tp_line = 'loband1'

            if direction is not None:
                can_enter = True

                # ========== ФИЛЬТР ПО СТОХАСТИКУ ==========
                if stoch_filter:
                    if direction == 'long':
                        if not (current_stoch < stoch_oversold):
                            can_enter = False
                    elif direction == 'short':
                        if not (current_stoch > stoch_overbought):
                            can_enter = False

                # Проверка минимального расстояния между сделками
                if len(trades) > 0 and (i - trades[-1]['exit_idx']) < lookback_min_bars:
                    can_enter = False

                # Проверка касания зелёной линии после стоп-лосса
                if can_enter and require_green_touch_after_sl and len(trades) > 0:
                    last_trade = trades[-1]
                    last_exit_idx = last_trade['exit_idx']
                    last_exit_type = last_trade['exit_type']

                    if last_exit_type == 'sl':
                        touched_green = False
                        for j in range(last_exit_idx + 1, i + 1):
                            low = df_back.iloc[j]['low']
                            high = df_back.iloc[j]['high']
                            loband1_j = df_back.iloc[j]['loband1']
                            upband1_j = df_back.iloc[j]['upband1']
                            if (low <= loband1_j <= high) or (low <= upband1_j <= high):
                                touched_green = True
                                break
                        if not touched_green:
                            can_enter = False

                if can_enter:
                    amount = (capital_per_trade * (1 - commission)) / entry_price

                    if direction == 'long':
                        distance_to_tp = tp_price - entry_price
                        sl_price = entry_price - distance_to_tp / sl_ratio

                        position = {
                            'type': 'long',
                            'entry_idx': i,
                            'entry_price': entry_price,
                            'amount': amount,
                            'tp': tp_price,
                            'initial_tp': tp_price,
                            'sl': sl_price,
                            'initial_sl': sl_price,
                            'tp_trailing_active': trailing_tp,
                            'tp_highest_price': entry_price,
                            'tp_line': tp_line,
                            'tp_mode': tp_mode,
                            'trailing_activated': False
                        }
                        df_back.at[df_back.index[i], 'signal'] = 1
                        df_back.at[df_back.index[i], 'entry_price'] = entry_price

                    else:  # short
                        distance_to_tp = entry_price - tp_price
                        sl_price = entry_price + distance_to_tp / sl_ratio

                        position = {
                            'type': 'short',
                            'entry_idx': i,
                            'entry_price': entry_price,
                            'amount': amount,
                            'tp': tp_price,
                            'initial_tp': tp_price,
                            'sl': sl_price,
                            'initial_sl': sl_price,
                            'tp_trailing_active': trailing_tp,
                            'tp_lowest_price': entry_price,
                            'tp_line': tp_line,
                            'tp_mode': tp_mode,
                            'trailing_activated': False
                        }
                        df_back.at[df_back.index[i], 'signal'] = -1
                        df_back.at[df_back.index[i], 'entry_price'] = entry_price

                    continue

        # ========== ВЫХОД ИЗ ПОЗИЦИИ ==========
        if position is not None:

            # Сохраняем базовый динамический TP до возможных изменений
            base_dynamic_tp = None

            # Динамическое обновление TP
            if position['tp_mode'] == 'dynamic':
                if position['type'] == 'long':
                    if position['tp_line'] == 'upband1':
                        base_dynamic_tp = upband1
                    elif position['tp_line'] == 'meanline':
                        base_dynamic_tp = meanline
                    else:
                        base_dynamic_tp = loband1
                else:  # short
                    if position['tp_line'] == 'loband1':
                        base_dynamic_tp = loband1
                    elif position['tp_line'] == 'meanline':
                        base_dynamic_tp = meanline
                    else:
                        base_dynamic_tp = upband1

                # Если трейлинг ещё не активирован, используем базовый динамический TP
                if not position.get('trailing_activated', False):
                    position['tp'] = base_dynamic_tp

            # ========== ТРЕЙЛИНГ ДЛЯ FIXED РЕЖИМА ==========
            if position['tp_mode'] == 'fixed' and position['tp_trailing_active']:
                if position['type'] == 'long':
                    # Обновляем максимум
                    if current_high > position['tp_highest_price']:
                        position['tp_highest_price'] = current_high
                        new_tp = position['tp_highest_price'] * (1 - trailing_tp_distance / 100)
                        if new_tp > position['initial_tp']:
                            position['tp'] = max(position['tp'], new_tp)
                else:  # short
                    # Обновляем минимум
                    if current_low < position['tp_lowest_price']:
                        position['tp_lowest_price'] = current_low
                        new_tp = position['tp_lowest_price'] * (1 + trailing_tp_distance / 100)
                        if new_tp < position['initial_tp']:
                            position['tp'] = min(position['tp'], new_tp)

            # ========== ТРЕЙЛИНГ ДЛЯ DYNAMIC РЕЖИМА (АКТИВАЦИЯ) ==========
            if (position['tp_mode'] == 'dynamic' and
                position['tp_trailing_active'] and
                not position.get('trailing_activated', False)):

                # Проверяем касание базового TP
                if position['type'] == 'long':
                    if current_high >= base_dynamic_tp:
                        position['trailing_activated'] = True
                        position['tp_highest_price'] = max(position['tp_highest_price'], current_high)
                        new_tp = position['tp_highest_price'] * (1 - trailing_tp_distance / 100)
                        position['tp'] = max(base_dynamic_tp, new_tp)
                else:  # short
                    if current_low <= base_dynamic_tp:
                        position['trailing_activated'] = True
                        position['tp_lowest_price'] = min(position['tp_lowest_price'], current_low)
                        new_tp = position['tp_lowest_price'] * (1 + trailing_tp_distance / 100)
                        position['tp'] = min(base_dynamic_tp, new_tp)

            # ========== ОБНОВЛЕНИЕ ТРЕЙЛИНГА ПОСЛЕ АКТИВАЦИИ (DYNAMIC) ==========
            if (position['tp_mode'] == 'dynamic' and
                position.get('trailing_activated', False) and
                position['tp_trailing_active']):

                if position['type'] == 'long':
                    if current_high > position['tp_highest_price']:
                        position['tp_highest_price'] = current_high
                        new_tp = position['tp_highest_price'] * (1 - trailing_tp_distance / 100)
                        current_base_tp = base_dynamic_tp if base_dynamic_tp is not None else position['initial_tp']
                        position['tp'] = max(current_base_tp, new_tp)
                else:  # short
                    if current_low < position['tp_lowest_price']:
                        position['tp_lowest_price'] = current_low
                        new_tp = position['tp_lowest_price'] * (1 + trailing_tp_distance / 100)
                        current_base_tp = base_dynamic_tp if base_dynamic_tp is not None else position['initial_tp']
                        position['tp'] = min(current_base_tp, new_tp)

            # ========== ПРОВЕРКА УСЛОВИЙ ВЫХОДА ==========
            exit_price = None
            exit_type = None

            if position['type'] == 'long':
                if current_high >= position['tp']:
                    exit_price = position['tp']
                    exit_type = 'tp_trailing' if position.get('trailing_activated', False) or (position['tp_mode'] == 'fixed' and position['tp_trailing_active']) else 'tp'
                elif current_low <= position['sl']:
                    exit_price = position['sl']
                    exit_type = 'sl'

            else:  # short
                if current_low <= position['tp']:
                    exit_price = position['tp']
                    exit_type = 'tp_trailing' if position.get('trailing_activated', False) or (position['tp_mode'] == 'fixed' and position['tp_trailing_active']) else 'tp'
                elif current_high >= position['sl']:
                    exit_price = position['sl']
                    exit_type = 'sl'

            if exit_price is not None:
                if position['type'] == 'long':
                    exit_price_adj = exit_price * (1 - slippage)
                    exit_proceeds = position['amount'] * exit_price_adj * (1 - commission)
                    pnl = exit_proceeds - capital_per_trade
                else:
                    exit_price_adj = exit_price * (1 + slippage)
                    entry_proceeds = position['amount'] * position['entry_price'] * (1 - commission)
                    exit_cost = position['amount'] * exit_price_adj * (1 + commission)
                    pnl = entry_proceeds - exit_cost

                trades.append({
                    'entry_idx': position['entry_idx'],
                    'exit_idx': i,
                    'type': position['type'],
                    'entry_price': position['entry_price'],
                    'exit_price': exit_price_adj,
                    'pnl': pnl,
                    'exit_type': exit_type
                })

                df_back.at[df_back.index[i], 'exit_price'] = exit_price_adj
                df_back.at[df_back.index[i], 'pnl'] = pnl
                df_back.at[df_back.index[i], 'trade_type'] = f"{position['type']}_{exit_type}"

                position = None

    # ========== ПРИНУДИТЕЛЬНОЕ ЗАКРЫТИЕ ==========
    if position is not None:
        last_price = df_back.iloc[-1]['close']

        if position['type'] == 'long':
            exit_price_adj = last_price * (1 - slippage)
            exit_proceeds = position['amount'] * exit_price_adj * (1 - commission)
            pnl = exit_proceeds - capital_per_trade
        else:
            exit_price_adj = last_price * (1 + slippage)
            entry_proceeds = position['amount'] * position['entry_price'] * (1 - commission)
            exit_cost = position['amount'] * exit_price_adj * (1 + commission)
            pnl = entry_proceeds - exit_cost

        trades.append({
            'entry_idx': position['entry_idx'],
            'exit_idx': len(df_back) - 1,
            'type': position['type'],
            'entry_price': position['entry_price'],
            'exit_price': exit_price_adj,
            'pnl': pnl,
            'exit_type': 'forced'
        })

        df_back.at[df_back.index[-1], 'exit_price'] = exit_price_adj
        df_back.at[df_back.index[-1], 'pnl'] = pnl
        df_back.at[df_back.index[-1], 'trade_type'] = f"{position['type']}_forced"

    # ========== РАСЧЁТ МЕТРИК ==========
    trades_df = pd.DataFrame(trades)

    if len(trades_df) == 0:
        print("Нет сделок.")
        return df_back, {
            'total_pnl': 0,
            'avg_return_per_trade_percent': 0,
            'win_rate': 0,
            'num_trades': 0,
            'avg_win': 0,
            'avg_loss': 0,
            'max_drawdown': 0,
            'sharpe_approx': 0
        }

    total_pnl = trades_df['pnl'].sum()
    avg_return_per_trade_percent = (total_pnl / (capital_per_trade * len(trades_df))) * 100
    win_rate = (trades_df['pnl'] > 0).mean()
    num_trades = len(trades_df)
    avg_win = trades_df[trades_df['pnl'] > 0]['pnl'].mean() if (trades_df['pnl'] > 0).any() else 0.0
    avg_loss = trades_df[trades_df['pnl'] < 0]['pnl'].mean() if (trades_df['pnl'] < 0).any() else 0.0

    cum_pnl = trades_df['pnl'].cumsum()
    running_max = cum_pnl.cummax()
    max_drawdown = (running_max - cum_pnl).max()

    returns = trades_df['pnl'] / capital_per_trade
    sharpe = (returns.mean() / returns.std()) * np.sqrt(252) if len(returns) > 1 and returns.std() != 0 else 0.0

    metrics = {
        'total_pnl': total_pnl,
        'avg_return_per_trade_percent': avg_return_per_trade_percent,
        'win_rate': win_rate,
        'num_trades': num_trades,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'max_drawdown': max_drawdown,
        'sharpe_approx': sharpe
    }

    return df_back, metrics

In [ ]:
%load_ext autoreload
%autoreload

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import SRC.LIBRARIES.new_indicator_plot_utils as nipu

# ========== НАСТРОЙКИ ОТОБРАЖЕНИЯ ==========
is_log_scale_by_default = True
candlestick_row = 1

# ========== СОЗДАНИЕ ГРАФИКА ==========
fig_main = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0.03)

fig_main.add_trace(
    go.Candlestick(
        x=df[_KIEV_TIMESTAMP],
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="OHLC"
    ),
    row=candlestick_row, col=1
)

nipu.add_mrc(candlestick_row, fig_main, df)

# ========== ЗАПУСК БЭКТЕСТА ==========
df_backtest, metrics = backtest_red_cross(
    df,
    commission=commission_rate,
    capital_per_trade=capital_per_trade,
    slippage=0.0002,
    lookback_min_bars=LOOKBACK_MIN_BARS,
    tp_level=TP_LEVEL,
    sl_ratio=SL_RATIO,
    tp_mode=TP_MODE,
    trailing_tp=TRAILING_TP,
    trailing_tp_distance=TRAILING_TP_DISTANCE,
    require_green_touch_after_sl=REQUIRE_GREEN_TOUCH_AFTER_SL,
    min_bars_outside=MIN_BARS_OUTSIDE,
    stoch_filter=STOCH_FILTER,
    stoch_series=STOCH_SERIES,
    stoch_oversold=STOCH_OVERSOLD,
    stoch_overbought=STOCH_OVERBOUGHT
)

# ========== ВЫВОД РЕЗУЛЬТАТОВ ==========
print(f"Результаты стратегии (обратное пересечение красной линии):")
print(f"{sym} {discretization} | {display_start_date_str} - {display_end_date_str}")
print(f"TP уровень: {TP_LEVEL}")
print(f"TP режим: {TP_MODE}")
print(f"SL соотношение: 1:{SL_RATIO}")
print(f"Требовать касание зелёной после выхода: {REQUIRE_GREEN_TOUCH_AFTER_SL}")
print(f"Трейлинг-ТП: включён={TRAILING_TP}, отступ={TRAILING_TP_DISTANCE}%")
print(f"Фильтр по стохастику: {STOCH_FILTER} (серия={STOCH_SERIES}, oversold={STOCH_OVERSOLD}, overbought={STOCH_OVERBOUGHT})")
print("-" * 40)

for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

# ========== НАСТРОЙКИ ГРАФИКА ==========
price_log_scale_value = "log"
price_linear_scale_value = "linear"
price_log_scale_title = "Price (log scale)"
price_linear_scale_title = "Price (linear scale)"

fig_main.update_layout(
    title=f"🐵 {discretization} | TP={TP_LEVEL}({TP_MODE}) | SL=1:{SL_RATIO}",
    xaxis_rangeslider_visible=False,
    yaxis1_type=price_log_scale_value if is_log_scale_by_default else price_linear_scale_value,
    yaxis1_title=price_log_scale_title if is_log_scale_by_default else price_linear_scale_title,
    height=1200,
    bargap=0,
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            active=1 if is_log_scale_by_default else 0,
            x=0.315,
            y=1.073,
            buttons=[
                dict(
                    label="Linear",
                    method="relayout",
                    args=[{"yaxis.type": price_linear_scale_value, "yaxis.title.text": price_linear_scale_title}]
                ),
                dict(
                    label="Log",
                    method="relayout",
                    args=[{"yaxis.type": price_log_scale_value, "yaxis.title.text": price_log_scale_title}]
                )
            ],
            font=dict(color="red", size=12, family="Arial"),
            bgcolor="rgba(0, 0, 0, 0)",
            bordercolor="red",
            borderwidth=1
        )
    ]
)

# ========== МАРКЕРЫ СДЕЛОК ==========
entries_long = df_backtest[df_backtest['signal'] == 1].index
entries_short = df_backtest[df_backtest['signal'] == -1].index

exits_tp = df_backtest[
    df_backtest['trade_type'].str.contains('tp', na=False) &
    ~df_backtest['trade_type'].str.contains('trailing', na=False)
].index

exits_tp_trailing = df_backtest[df_backtest['trade_type'].str.contains('tp_trailing', na=False)].index
exits_sl = df_backtest[df_backtest['trade_type'].str.contains('sl', na=False)].index

# Long Entry
fig_main.add_trace(
    go.Scatter(
        x=df_backtest.loc[entries_long, _KIEV_TIMESTAMP],
        y=df_backtest.loc[entries_long, 'entry_price'],
        mode='markers',
        name='Long Entry',
        marker=dict(symbol='triangle-up', size=10, color='green')
    ),
    row=candlestick_row, col=1
)

# Short Entry
fig_main.add_trace(
    go.Scatter(
        x=df_backtest.loc[entries_short, _KIEV_TIMESTAMP],
        y=df_backtest.loc[entries_short, 'entry_price'],
        mode='markers',
        name='Short Entry',
        marker=dict(symbol='triangle-down', size=10, color='red')
    ),
    row=candlestick_row, col=1
)

# Take Profit
fig_main.add_trace(
    go.Scatter(
        x=df_backtest.loc[exits_tp, _KIEV_TIMESTAMP],
        y=df_backtest.loc[exits_tp, 'exit_price'],
        mode='markers',
        name='Take Profit',
        marker=dict(
            symbol='circle',
            size=10,
            color='lime',
            opacity=0.7,
            line=dict(width=1, color='darkgreen')
        )
    ),
    row=candlestick_row, col=1
)

# TP (Trailing)
fig_main.add_trace(
    go.Scatter(
        x=df_backtest.loc[exits_tp_trailing, _KIEV_TIMESTAMP],
        y=df_backtest.loc[exits_tp_trailing, 'exit_price'],
        mode='markers',
        name='TP (Trailing)',
        marker=dict(
            symbol='diamond',
            size=12,
            color='lightgreen',
            opacity=0.8,
            line=dict(width=1, color='green')
        )
    ),
    row=candlestick_row, col=1
)

# Stop Loss
fig_main.add_trace(
    go.Scatter(
        x=df_backtest.loc[exits_sl, _KIEV_TIMESTAMP],
        y=df_backtest.loc[exits_sl, 'exit_price'],
        mode='markers',
        name='Stop Loss',
        marker=dict(
            symbol='x',
            size=12,
            color='red',
            line=dict(width=2, color='darkred')
        )
    ),
    row=candlestick_row, col=1
)

fig_main.show()

# Звуковой сигнал
os.system('afplay /System/Library/Sounds/Glass.aiff')
print('\a')